# SkyGuard AI — original-observation dataset builder

This **online, CPU-only** builder acquires NOAA SURFRAD files published for
Bondville (`bon`), Fort Peck (`fpk`) and Goodwin Creek (`gwn`) in 2023–2024.
The provider documents temperature, independently measured relative humidity,
and station pressure. These US stations are a sensor-method development corpus;
they do not establish performance on Indian AWS.

The attached `skyguard-sih26073` source-only release is verified before import
or acquisition. The adapter retains native daily files, HTTP receipts, immutable
hashes, field provenance, quality codes and observed minute-00 records. Missing
readings stay missing and all hardware-fault labels stay unknown. The independent
preflight verifies the processed records against archived original observations.

The release chain remains: local source bundle → `skyguard-sih26073` → this
online builder → `skyguard-sih26073-processed` → offline CPU training. The output
is one complete directory, with source and original evidence. Publish the
directory after this notebook succeeds. Runtime depends on provider throughput;
an incomplete acquisition or failed data verification stops the build.


## 1 · Verify the attached source release

In [ ]:
CODE_MANIFEST = {'schema_version': 1, 'role': 'expected_processed_training_source', 'sha256': 'bcaf2e25822dca8f4dddb7331af46ef6b2e975e78f927c49de485fcbb096c50e', 'files': {'src/awsad/__init__.py': '314c2100758b0ca67370ab7fae387f732dfdd5e90404c07c7a5cbba88a6ce559', 'src/awsad/data/acquisition.py': 'a221495715cf6630b9335c3fc5772485fc1c80ee89403903c462836d22efa392', 'src/awsad/data/fetch_surfrad.py': 'c44c0e5a05e8f1973b77877e28e3d52e6fe29efe0b5e31c3ecfa8f26a1c61543', 'src/awsad/data/surfrad.py': 'd388939c654bf9ad8942773efe8fde86bd44fb05c4271d04f3cacf241c8d5d6f', 'src/awsad/data/training_contract.py': '789f2349167bcd366020199011ae96866c8d74eb0006190f7a7436f5f9f68e4d', 'src/awsad/data/verify_surfrad.py': 'a978ea318d535dce11aa82ade817bece82d85386e43b63f840347be2b86a0131', 'src/awsad/real_observation_pipeline.py': '7a438153e20ad66422d7ab6bc93c4d3af420f5d46dfcf39332092371f67d5c3e'}}
import hashlib, importlib, json, pathlib, shutil, sys, tempfile, zipfile
from datetime import datetime, timezone
sys.dont_write_bytecode = True
WORKDIR = pathlib.Path("/kaggle/working")
INPUTDIR = pathlib.Path("/kaggle/input")
WORKDIR.mkdir(parents=True, exist_ok=True)

def _source_manifest(raw):
    if len(raw) > 2_000_000:
        raise RuntimeError("source manifest exceeds size limit")
    blob = json.loads(raw)
    return blob if isinstance(blob, dict) and blob.get("bundle_role") == "online_builder_source" else None

def _relative_path(rel):
    if not isinstance(rel, str) or not rel or "\\" in rel or ":" in rel:
        raise RuntimeError("unsafe source path: " + str(rel))
    path = pathlib.PurePosixPath(rel)
    if path.is_absolute() or ".." in path.parts or path.as_posix() != rel or rel == ".":
        raise RuntimeError("unsafe source path: " + rel)
    return path

def _sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()

def _verify_files(root, hashes):
    if not isinstance(hashes, dict) or not hashes or "MANIFEST.json" in hashes:
        raise RuntimeError("source requires a complete SHA-256 map excluding MANIFEST.json")
    resolved_root = root.resolve()
    for rel, expected in hashes.items():
        parts = _relative_path(rel)
        if not isinstance(expected, str) or len(expected) != 64 or any(c not in "0123456789abcdef" for c in expected):
            raise RuntimeError("invalid source SHA-256: " + rel)
        path = root.joinpath(*parts.parts)
        if path.is_symlink() or resolved_root not in path.resolve().parents or not path.is_file():
            raise RuntimeError("missing or unsafe source file: " + rel)
        if _sha256(path) != expected:
            raise RuntimeError("source bundle hash mismatch: " + rel)
    paths = list(root.rglob("*"))
    if any(path.is_symlink() for path in paths):
        raise RuntimeError("source bundle must not contain symbolic links")
    actual = {path.relative_to(root).as_posix() for path in paths
              if path.is_file() and path != root / "MANIFEST.json"}
    if actual != set(hashes):
        raise RuntimeError("source manifest must cover exactly the retained source files")

candidates = []
for path in sorted(INPUTDIR.glob("**/MANIFEST.json")):
    if not (path.parent / "src/awsad/__init__.py").is_file():
        continue  # Archived documentation manifests are not attached source roots.
    if path.stat().st_size > 2_000_000:
        continue
    raw = path.read_bytes()
    try:
        manifest = _source_manifest(raw)
    except (ValueError, UnicodeError):
        continue
    if manifest:
        candidates.append({"kind": "root", "path": path.parent, "prefix": "", "raw": raw, "manifest": manifest})
for path in sorted(INPUTDIR.glob("**/*.zip")):
    try:
        with zipfile.ZipFile(path) as zf:
            names = set(zf.namelist())
            for member in zf.infolist():
                if not member.filename.endswith("MANIFEST.json") or member.file_size > 2_000_000:
                    continue
                prefix = member.filename[:-len("MANIFEST.json")]
                if prefix + "src/awsad/__init__.py" not in names:
                    continue  # An archived manifest without adjacent source is evidence only.
                raw = zf.read(member)
                try:
                    manifest = _source_manifest(raw)
                except (ValueError, UnicodeError):
                    continue
                if manifest:
                    if prefix:
                        _relative_path(prefix.rstrip("/"))
                    candidates.append({"kind": "zip", "path": path, "prefix": prefix, "raw": raw, "manifest": manifest})
    except zipfile.BadZipFile:
        continue
if not candidates:
    raise FileNotFoundError("Attach the current skyguard-sih26073 source-only dataset containing MANIFEST.json")
identities = {hashlib.sha256(candidate["raw"]).hexdigest() for candidate in candidates}
if len(identities) != 1:
    raise RuntimeError("Ambiguous source inputs; attach exactly one source release")
chosen = sorted(candidates, key=lambda candidate: (candidate["kind"] != "root", str(candidate["path"])))[0]
SOURCE_MANIFEST = chosen["manifest"]
SOURCE_MANIFEST_BYTES = chosen["raw"]
SOURCE_MANIFEST_SHA256 = hashlib.sha256(SOURCE_MANIFEST_BYTES).hexdigest()
if (SOURCE_MANIFEST.get("schema_version") != 1
        or SOURCE_MANIFEST.get("processed_data_included") is not False
        or SOURCE_MANIFEST.get("training_data_policy") != "real_observations_only"):
    raise RuntimeError("Expected a schema-1, real-observations-only, source-only release")
if chosen["kind"] == "root":
    SOURCE_ROOT = chosen["path"]
else:
    SOURCE_ROOT = pathlib.Path(tempfile.mkdtemp(prefix="skyguard-source-"))
    extracted = set()
    with zipfile.ZipFile(chosen["path"]) as zf:
        for member in zf.infolist():
            if not member.filename.startswith(chosen["prefix"]):
                continue
            rel = member.filename[len(chosen["prefix"]):].rstrip("/")
            if not rel:
                continue
            parts = _relative_path(rel)
            if ((member.external_attr >> 16) & 0o170000) == 0o120000:
                raise RuntimeError("source archive must not contain symbolic links")
            destination = SOURCE_ROOT.joinpath(*parts.parts)
            if member.is_dir():
                destination.mkdir(parents=True, exist_ok=True)
            else:
                if rel in extracted:
                    raise RuntimeError("duplicate source archive member: " + rel)
                extracted.add(rel)
                destination.parent.mkdir(parents=True, exist_ok=True)
                with zf.open(member) as src, destination.open("wb") as dst:
                    shutil.copyfileobj(src, dst)
_verify_files(SOURCE_ROOT, SOURCE_MANIFEST.get("sha256"))
mismatches = [rel for rel, expected in CODE_MANIFEST["files"].items()
              if SOURCE_MANIFEST["sha256"].get(rel) != expected]
if mismatches:
    raise RuntimeError("Stale or missing source; regenerate source bundle and both notebooks: " + repr(mismatches))
for module_name in list(sys.modules):
    if module_name == "awsad" or module_name.startswith("awsad."):
        del sys.modules[module_name]
sys.path.insert(0, str(SOURCE_ROOT / "src"))
importlib.invalidate_caches()
import awsad
assert pathlib.Path(awsad.__file__).resolve() == (SOURCE_ROOT / "src/awsad/__init__.py").resolve()
BUILD_UTC = datetime.now(timezone.utc)
BUILD_ID = BUILD_UTC.strftime("%Y%m%dT%H%M%S%fZ") + "-" + SOURCE_MANIFEST_SHA256[:12]
OUT = WORKDIR / ("awsad_real_bundle_" + BUILD_ID)
OUT.mkdir()
ARCHIVE = pathlib.Path(tempfile.mkdtemp(prefix="skyguard-surfrad-"))
print("Verified source release:", SOURCE_MANIFEST_SHA256)
print("Expected training source:", CODE_MANIFEST["sha256"])
print("Output directory:", OUT)


## 2 · Acquire published original records

Every daily URL must first appear in the provider inventory. Four concurrent requests are the maximum. Network failures are recorded and cannot become invented rows.

In [ ]:
from awsad.data.fetch_surfrad import fetch_surfrad
STATIONS = ("bon", "fpk", "gwn")
YEARS = (2023, 2024)
inventory = fetch_surfrad(ARCHIVE, stations=STATIONS, years=YEARS, workers=4)
if not inventory.get("completed_at_utc") or inventory.get("failures") or not inventory.get("files"):
    raise RuntimeError("Acquisition is incomplete; inspect surfrad_acquisition.json before rebuilding")
print("Acquired published daily files:", len(inventory["files"]))


## 3 · Build and independently verify observations

Select existing minute-00 records; do not interpolate, aggregate, replace missing measurements or manufacture fault labels.

In [ ]:
from awsad.data.surfrad import build_surfrad_dataset
from awsad.data.training_contract import require_eligible_training_data
PROCESSED = OUT / "data" / "processed"
metadata = build_surfrad_dataset(ARCHIVE, PROCESSED, station_codes=STATIONS, years=YEARS)
preflight = require_eligible_training_data(PROCESSED, report_path=OUT / "data_preflight.json")
if not preflight.get("eligible"):
    raise RuntimeError("Independent raw-to-processed verification did not accept this dataset")
print("Original provider observation rows:", metadata["rows"])
print("Data eligibility:", preflight["eligible"])
print("Hardware fault labels remain unknown; provider quality flags are retained separately.")


## 4 · Preserve the release chain and publishable output

The final manifest covers every retained file. Original provider evidence and the exact source release travel with the processed dataset.

In [ ]:
# Preserve the adapter's internal metadata and evidence hashes unchanged.
_verify_files(SOURCE_ROOT, SOURCE_MANIFEST["sha256"])
for rel in SOURCE_MANIFEST["sha256"]:
    destination = OUT.joinpath(*_relative_path(rel).parts)
    if destination.exists():
        raise RuntimeError("Source release collides with built data: " + rel)
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(SOURCE_ROOT / rel, destination)
provenance = OUT / "provenance"
provenance.mkdir(exist_ok=True)
if (provenance / "source_manifest.json").exists() or (provenance / "expected_source.json").exists():
    raise RuntimeError("Source release collides with reserved provenance paths")
(provenance / "source_manifest.json").write_bytes(SOURCE_MANIFEST_BYTES)
(provenance / "expected_source.json").write_text(json.dumps(CODE_MANIFEST, indent=2) + "\n", encoding="utf-8")
hashes = {path.relative_to(OUT).as_posix(): _sha256(path) for path in sorted(OUT.rglob("*")) if path.is_file()}
bundle_manifest = {
    "schema_version": 1, "project": "SkyGuard AI / SIH26073", "bundle_role": "processed_training_bundle",
    "training_data_policy": "real_observations_only", "build_id": BUILD_ID,
    "build_utc": BUILD_UTC.isoformat(), "source_manifest_sha256": SOURCE_MANIFEST_SHA256,
    "training_source_sha256": CODE_MANIFEST["sha256"], "source": "noaa_surfrad",
    "stations": list(STATIONS), "years": list(YEARS), "observed_rows": metadata["rows"],
    "unknown_fault_labels": True, "processed_directory": "data/processed",
    "raw_observations_retained": True, "sha256": hashes,
}
(OUT / "MANIFEST.json").write_text(json.dumps(bundle_manifest, indent=2) + "\n", encoding="utf-8")
_verify_files(OUT, hashes)
print("Verified output files:", len(hashes))
print("Publish this output directory as skyguard-sih26073-processed:", OUT)
print("Then run the matching offline CPU training notebook.")
